# Huấn Luyện EfficientNet-B0 Nhận Diện & Chẩn Đoán Bệnh 4 Loại Trái Cây (17 Lớp)

Notebook này triển khai giải pháp chuyển giao học tập (Transfer Learning & Fine-Tuning) với mô hình **`EfficientNet-B0`** trên bộ dữ liệu 4 loại cây ăn quả: **Táo (Apple), Ổi (Guava), Xoài (Mango), Lựu (Pomegranate)** bao gồm 17 trạng thái bệnh học và quả khỏe mạnh.

---

### 1. Thông Tin Bộ Dữ Liệu Nguồn
- **Nguồn Kaggle**: [`ateebnoone/fruits-dataset-for-fruit-disease-classification`](https://www.kaggle.com/datasets/ateebnoone/fruits-dataset-for-fruit-disease-classification)
- **Tham chiếu kiến trúc**: [`ateebnoone/fruit-disease-classification-01`](https://www.kaggle.com/code/ateebnoone/fruit-disease-classification-01)
- **17 Lớp Phân Loại**:
  - 🍎 **Táo (4 lớp)**: `APPLE_Blotch_Apple`, `APPLE_Healthy_Apple`, `APPLE_Rot_Apple`, `APPLE_Scab_Apple`
  - 🍈 **Ổi (3 lớp)**: `GUAVA_Anthracnose_Guava`, `GUAVA_Fruitfly_Guava`, `GUAVA_Healthy_Guava`
  - 🥭 **Xoài (5 lớp)**: `MANGO_Alternaria_Mango`, `MANGO_Anthracnose_Mango`, `MANGO_Black Mould Rot (Aspergillus)_Mango`, `MANGO_Healthy_Mango`, `MANGO_Stem and Rot (Lasiodiplodia)_Mango`
  - 🫐 **Lựu (5 lớp)**: `POMEGRANATE_Alternaria_Pomegranate`, `POMEGRANATE_Anthracnose_Pomegranate`, `POMEGRANATE_Bacterial_Blight_Pomegranate`, `POMEGRANATE_Cercospora_Pomegranate`, `POMEGRANATE_Healthy_Pomegranate`

---

### 2. Cấu Trúc Huấn Luyện Chuẩn Hóa
1. **Phân chia dữ liệu chuẩn**: **70% Train - 15% Validation - 15% Test** (phân tầng theo nhãn `stratify`, `seed=42`).
2. **Quy trình 2 pha (Two-Phase Transfer Learning)**:
   - *Pha 1 (Train Head)*: Khóa thân mạng EfficientNet-B0, huấn luyện phân loại (`lr=1e-3`).
   - *Pha 2 (Fine-Tuning)*: Mở khóa các tầng tích chập sâu cuối cùng với tốc độ học thấp (`lr=1e-4`), `ReduceLROnPlateau` và `EarlyStopping`.
3. **Đánh giá trên tập Test**: Classification Report, Confusion Matrix, và Cell kiểm thử suy luận (Inference + Grad-CAM + đo FPS).
4. **Xuất mô hình TFLite**: Lưu file `EfficientNetB0_model.tflite` sẵn sàng đưa lên Web demo.

In [ ]:
# CELL 1: Environment Setup & Hardware Sanity Check
import os
import sys
import time
import random
import shutil
import warnings
from pathlib import Path
from collections import defaultdict, Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

warnings.filterwarnings('ignore')

# Cố định Seed chuẩn mực
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

print(f"TensorFlow Version : {tf.__version__}")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Phát hiện {len(gpus)} GPU: {[g.name for g in gpus]}")
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"GPU Memory Growth  : Đã kích hoạt đồng bộ cho toàn bộ {len(gpus)} GPU")
    except Exception as e:
        print(f"GPU Config Warning : {e}")
else:
    print("Warning: Running on CPU (No GPU detected)!")

In [ ]:
# CELL 2: Dataset Path & Global Hyperparameters
import os
from pathlib import Path

# Đường dẫn ưu tiên số 1 theo người dùng chỉ định
USER_DATASET_PATH = Path("/kaggle/input/datasets/ateebnoone/fruits-dataset-for-fruit-disease-classification")
FALLBACK_DATASET_PATH = Path("/kaggle/input/fruits-dataset-for-fruit-disease-classification")

print("=== KIỂM TRA MÔI TRƯỜNG /kaggle/input ===")
if Path("/kaggle/input").exists():
    input_contents = list(Path("/kaggle/input").iterdir())
    print(f"Các mục trong /kaggle/input ({len(input_contents)} mục): {[p.name for p in input_contents]}")
else:
    print("Thư mục /kaggle/input chưa sẵn sàng.")

PROD_DATASET_PATH = None
if USER_DATASET_PATH.exists():
    PROD_DATASET_PATH = USER_DATASET_PATH
elif FALLBACK_DATASET_PATH.exists():
    PROD_DATASET_PATH = FALLBACK_DATASET_PATH
else:
    # Quét tìm kiếm tự động thư mục chứa 4 loại quả
    if Path("/kaggle/input").exists():
        candidates = [Path("/kaggle/input")] + list(Path("/kaggle/input").glob("*")) + list(Path("/kaggle/input").glob("*/*")) + list(Path("/kaggle/input").glob("*/*/*"))
        target_fruits = {"APPLE", "GUAVA", "MANGO", "POMEGRANATE"}
        for cand in candidates:
            if cand.is_dir():
                sub_names = {d.name.upper() for d in cand.iterdir() if d.is_dir()}
                if len(target_fruits.intersection(sub_names)) >= 3:
                    PROD_DATASET_PATH = cand
                    break

if PROD_DATASET_PATH is None:
    print("\n❌ LỖI: Dataset chưa được gắn vào phiên làm việc Kaggle!")
    print("👉 HƯỚNG DẪN XỬ LÝ NHANH TRÊN KAGGLE WEB:")
    print("1. Mở notebook: https://www.kaggle.com/code/nguynthanhthit/fruit-disease-classification-efficientnet-b0")
    print("2. Nhấn nút 'Edit' ở góc trên bên phải.")
    print("3. Ở thanh công cụ bên phải (Sidebar), nhấn '+ Add Input'.")
    print("4. Gõ tìm 'fruits-dataset-for-fruit-disease-classification' và nhấn nút '+' (Add).")
    print("5. Nhấn 'Save Version' hoặc 'Run All'.")
    raise AssertionError("Không tìm thấy thư mục dataset trong /kaggle/input.")

print(f"✅ Thư mục Dataset gốc xác định: {PROD_DATASET_PATH}")
found_fruits = [d.name for d in PROD_DATASET_PATH.iterdir() if d.is_dir()]
print(f"Các loại quả phát hiện: {found_fruits}")

IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32

INITIAL_EPOCHS = 20
FINE_TUNE_EPOCHS = 20
INITIAL_LR = 1e-3
FINE_TUNE_LR = 1e-4

print(f"Độ phân giải ảnh   : {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Kích thước Batch   : {BATCH_SIZE}")
print(f"Kế hoạch huấn luyện : {INITIAL_EPOCHS} Epochs (Phase 1) + {FINE_TUNE_EPOCHS} Epochs (Phase 2)")

In [ ]:
# CELL 3: Scan & Summarize Original Dataset Structure
image_extensions = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff', '.webp'}

fruit_categories = sorted([d.name for d in PROD_DATASET_PATH.iterdir() if d.is_dir()])
print(f"Tìm thấy {len(fruit_categories)} nhóm cây ăn quả: {fruit_categories}")

dataset_summary = []
all_filepaths = []

for fruit in fruit_categories:
    fruit_dir = PROD_DATASET_PATH / fruit
    conditions = sorted([d.name for d in fruit_dir.iterdir() if d.is_dir()])
    for cond in conditions:
        cond_dir = fruit_dir / cond
        class_name = f"{fruit}_{cond}"
        imgs = [f for f in cond_dir.iterdir() if f.is_file() and f.suffix.lower() in image_extensions]
        count = len(imgs)
        dataset_summary.append({
            "Loại Quả": fruit,
            "Tên Bệnh / Tình Trạng": cond,
            "Tên Lớp Chuẩn Hóa": class_name,
            "Số Lượng Ảnh": count
        })
        for img_path in imgs:
            all_filepaths.append((str(img_path), class_name))

df_summary = pd.DataFrame(dataset_summary)
print("=== BẢNG THỐNG KÊ CHI TIẾT 17 LỚP BỆNH TRÁI CÂY ===")
print(df_summary.to_string(index=False))
print(f"\nTổng số lớp phân loại : {len(df_summary)} lớp")
print(f"Tổng số ảnh thu thập   : {len(all_filepaths):,} ảnh")

In [ ]:
# CELL 4: Prepare Flattened Dataset Split with Stratified GroupShuffleSplit (0.00% Leakage)
import os
import re
import shutil
from pathlib import Path
from collections import defaultdict
import numpy as np
import pandas as pd
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess_input

flattened_dir = Path("flattened_dataset_split")
train_dir = flattened_dir / "train"
val_dir   = flattened_dir / "validation"
test_dir  = flattened_dir / "test"

if flattened_dir.exists():
    shutil.rmtree(flattened_dir)

train_dir.mkdir(parents=True, exist_ok=True)
val_dir.mkdir(parents=True, exist_ok=True)
test_dir.mkdir(parents=True, exist_ok=True)

# 1. Trích xuất Group ID cho từng ảnh dựa trên bản chất vật lý của từng quả
def extract_group_id(filepath, class_name):
    filename = Path(filepath).name
    # Ổi: Gom theo số ID quả đứng đầu (ví dụ: 10_unsharp_clahe_augmented_0.png -> Quả 10)
    if "GUAVA" in class_name.upper():
        m = re.match(r"^(\d+)_", filename)
        if m:
            return f"{class_name}_guava_fruit_{m.group(1)}"
    # Lựu: Gom theo chuỗi thời gian chụp IMG_YYYYMMDD_HHMMSS (loại bỏ ảnh chụp liên tiếp)
    elif "POMEGRANATE" in class_name.upper():
        m = re.search(r"(IMG_\d{8}_\d{6})", filename)
        if m:
            return f"{class_name}_pom_session_{m.group(1)}"
    # Táo: Gom theo index ảnh gốc
    elif "APPLE" in class_name.upper():
        m = re.search(r"_0_(\d+)", filename)
        if m:
            return f"{class_name}_apple_{m.group(1)}"
    # Xoài: Gom theo index ảnh gốc
    elif "MANGO" in class_name.upper():
        m = re.search(r"_(\d+)", filename)
        if m:
            return f"{class_name}_mango_{m.group(1)}"
    return f"{class_name}_{filename}"

samples_data = []
for f_path, c_name in all_filepaths:
    samples_data.append({
        "path": f_path,
        "class_name": c_name,
        "group_id": extract_group_id(f_path, c_name)
    })

df_samples = pd.DataFrame(samples_data)
all_classes = sorted(df_samples['class_name'].unique())

# 2. Phân chia GroupShuffleSplit theo từng lớp (70% Train, 15% Val, 15% Test)
rng = np.random.RandomState(SEED)
train_files = []
val_files = []
test_files = []

for cls in all_classes:
    cls_df = df_samples[df_samples['class_name'] == cls]
    unique_groups = cls_df['group_id'].unique()
    rng.shuffle(unique_groups)
    
    n_groups = len(unique_groups)
    n_test = max(1, int(round(n_groups * 0.15)))
    n_val  = max(1, int(round(n_groups * 0.15)))
    n_train = n_groups - n_test - n_val
    if n_train <= 0:
        n_train = max(1, n_groups - 2)
        n_val = 1
        n_test = 1
        
    test_groups = set(unique_groups[:n_test])
    val_groups  = set(unique_groups[n_test : n_test + n_val])
    train_groups = set(unique_groups[n_test + n_val:])
    
    # Kiểm tra tính toàn vẹn toán học (Tuyệt đối không rò rỉ)
    assert len(train_groups & val_groups) == 0, f"Rò rỉ Train-Val tại lớp {cls}!"
    assert len(train_groups & test_groups) == 0, f"Rò rỉ Train-Test tại lớp {cls}!"
    assert len(val_groups & test_groups) == 0, f"Rò rỉ Val-Test tại lớp {cls}!"
    
    train_files.extend(cls_df[cls_df['group_id'].isin(train_groups)][['path', 'class_name']].values.tolist())
    val_files.extend(cls_df[cls_df['group_id'].isin(val_groups)][['path', 'class_name']].values.tolist())
    test_files.extend(cls_df[cls_df['group_id'].isin(test_groups)][['path', 'class_name']].values.tolist())

print(f"=== KẾT QUẢ PHÂN CHIA STRATIFIED GROUP SPLIT (NO LEAKAGE) ===")
print(f"Tập Train      : {len(train_files):,} ảnh ({len(train_files)/len(df_samples)*100:.1f}%)")
print(f"Tập Validation : {len(val_files):,} ảnh ({len(val_files)/len(df_samples)*100:.1f}%)")
print(f"Tập Test       : {len(test_files):,} ảnh ({len(test_files)/len(df_samples)*100:.1f}%)")
print(f"Tổng số ảnh    : {len(df_samples):,} ảnh")
print(f"Xác nhận rò rỉ : 0.00% (Hoàn toàn không rò rỉ nhóm quả)")

# 3. Sao chép ảnh vào cấu trúc thư mục phẳng
for f_path, c_name in train_files:
    dest_dir = train_dir / c_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f_path, dest_dir / Path(f_path).name)

for f_path, c_name in val_files:
    dest_dir = val_dir / c_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f_path, dest_dir / Path(f_path).name)

for f_path, c_name in test_files:
    dest_dir = test_dir / c_name
    dest_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(f_path, dest_dir / Path(f_path).name)

print("Đã tạo thư mục phân chia dữ liệu phẳng thành công!")

# 4. Data Generators
train_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess_input,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

val_test_datagen = ImageDataGenerator(
    preprocessing_function=efficientnet_preprocess_input
)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=all_classes,
    shuffle=True,
    seed=SEED
)

validation_generator = val_test_datagen.flow_from_directory(
    val_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=all_classes,
    shuffle=False
)

test_generator = val_test_datagen.flow_from_directory(
    test_dir,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=all_classes,
    shuffle=False
)

In [ ]:
# CELL 5: Construct EfficientNet-B0 Transfer Learning Architecture
base_model = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, 3)
)

# Khóa thân mạng base_model ở giai đoạn đầu
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(name='global_avg_pool'),
    layers.BatchNormalization(name='bn_head'),
    layers.Dropout(0.40, name='dropout_1'),
    layers.Dense(512, activation='relu', name='dense_512'),
    layers.BatchNormalization(name='bn_dense'),
    layers.Dropout(0.30, name='dropout_2'),
    layers.Dense(num_classes, activation='softmax', name='classifier_head')
], name='Fruit_Disease_EfficientNetB0')

model.summary()

total_params = model.count_params()
trainable_params = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"\nTổng số tham số mạng : {total_params:,} (Trainable Phase 1: {trainable_params:,})")

In [ ]:
# CELL 6: Phase 1 - Train Classifier Head (Frozen Base)
os.makedirs("models", exist_ok=True)
phase1_checkpoint = "models/efficientnetb0_phase1_best.keras"

model.compile(
    optimizer=Adam(learning_rate=INITIAL_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

callbacks_phase1 = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(filepath=phase1_checkpoint, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1)
]

print(f"=== BẮT ĐẦU PHA 1: HUẤN LUYỆN HEAD ({INITIAL_EPOCHS} EPOCHS - LR: {INITIAL_LR}) ===")
history_p1 = model.fit(
    train_generator,
    epochs=INITIAL_EPOCHS,
    validation_data=val_generator,
    callbacks=callbacks_phase1
)

# Load lại trọng số tốt nhất của Pha 1
if os.path.exists(phase1_checkpoint):
    model.load_weights(phase1_checkpoint)
    print(f"Đã nạp trọng số tốt nhất Pha 1 từ {phase1_checkpoint}")

In [ ]:
# CELL 7: Phase 2 - Fine-Tuning (Unfreeze Top MBConv Layers)
# Mở khóa 20% các tầng cuối của EfficientNetB0
base_model.trainable = True
fine_tune_at = int(len(base_model.layers) * 0.80)

for layer in base_model.layers[:fine_tune_at]:
    layer.trainable = False

# Giữ nguyên BatchNormalization ở chế độ eval để bảo toàn thống kê ImageNet
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

phase2_checkpoint = "models/efficientnetb0_final_finetuned.keras"

model.compile(
    optimizer=Adam(learning_rate=FINE_TUNE_LR),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

trainable_params_ft = sum([tf.size(w).numpy() for w in model.trainable_weights])
print(f"Số lượng tầng mở khóa Fine-Tuning: {len(base_model.layers) - fine_tune_at} tầng")
print(f"Số tham số huấn luyện Pha 2       : {trainable_params_ft:,}")

callbacks_phase2 = [
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(filepath=phase2_checkpoint, monitor='val_accuracy', save_best_only=True, mode='max', verbose=1),
    EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True, verbose=1)
]

initial_epochs_run = len(history_p1.history['loss'])

print(f"\n=== BẮT ĐẦU PHA 2: FINE-TUNING ({FINE_TUNE_EPOCHS} EPOCHS - LR: {FINE_TUNE_LR}) ===")
history_p2 = model.fit(
    train_generator,
    epochs=initial_epochs_run + FINE_TUNE_EPOCHS,
    initial_epoch=initial_epochs_run,
    validation_data=val_generator,
    callbacks=callbacks_phase2
)

# Nạp lại trọng số tối ưu nhất
if os.path.exists(phase2_checkpoint):
    model.load_weights(phase2_checkpoint)
    print(f"Đã nạp checkpoint tốt nhất toàn diện từ {phase2_checkpoint}")

In [ ]:
# CELL 8: Plot Combined Two-Phase Training Curves
loss = history_p1.history['loss'] + history_p2.history['loss']
val_loss = history_p1.history['val_loss'] + history_p2.history['val_loss']
acc = history_p1.history['accuracy'] + history_p2.history['accuracy']
val_acc = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']

epochs_range = range(1, len(loss) + 1)
plt.figure(figsize=(15, 5))

# Loss Plot
plt.subplot(1, 2, 1)
plt.plot(epochs_range, loss, label='Train Loss', color='crimson', lw=2)
plt.plot(epochs_range, val_loss, label='Val Loss', color='dodgerblue', lw=2)
plt.axvline(x=initial_epochs_run + 0.5, color='gray', linestyle='--', label='Bắt đầu Fine-Tuning')
plt.title('Đường Cong Mất Mát (Loss) Qua 2 Pha', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Loss', fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

# Accuracy Plot
plt.subplot(1, 2, 2)
plt.plot(epochs_range, [a * 100 for a in acc], label='Train Accuracy', color='forestgreen', lw=2)
plt.plot(epochs_range, [a * 100 for a in val_acc], label='Val Accuracy', color='purple', lw=2)
plt.axvline(x=initial_epochs_run + 0.5, color='gray', linestyle='--', label='Bắt đầu Fine-Tuning')
plt.title('Đường Cong Độ Chính Xác (Accuracy %) Qua 2 Pha', fontsize=13, fontweight='bold')
plt.xlabel('Epochs', fontsize=11)
plt.ylabel('Accuracy (%)', fontsize=11)
plt.grid(True, alpha=0.3)
plt.legend(fontsize=10)

plt.tight_layout()
plt.savefig("learning_curves_efficientnetb0.png", dpi=150)
plt.show()

In [ ]:
# CELL 9: Comprehensive Test Evaluation & Metrics Report
test_loss, test_acc = model.evaluate(test_generator, verbose=1)
print(f"\n=================================================================")
print(f"   KẾT QUẢ ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP ({test_generator.samples} ẢNH)")
print(f"=================================================================")
print(f"  - Test Loss     : {test_loss:.4f}")
print(f"  - Test Accuracy : {test_acc * 100:.2f}%")
print(f"=================================================================\n")

# Dự đoán toàn bộ tập Test
test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred_classes = np.argmax(y_pred_probs, axis=1)
y_true_classes = test_generator.classes

# Báo cáo phân loại chuẩn
print("BÁO CÁO PHÂN LOẠI CHI TIẾT 17 LỚP (CLASSIFICATION REPORT):")
cls_report = classification_report(
    y_true_classes, y_pred_classes, target_names=all_classes, digits=4
)
print(cls_report)

# Ma trận nhầm lẫn chuẩn hóa
cm = confusion_matrix(y_true_classes, y_pred_classes)
cm_norm = cm.astype('float') / (cm.sum(axis=1)[:, np.newaxis] + 1e-10)

plt.figure(figsize=(14, 12))
sns.heatmap(
    cm_norm, annot=True, fmt='.2f', cmap='Greens',
    xticklabels=all_classes, yticklabels=all_classes,
    annot_kws={'size': 9}
)
plt.title(f'Ma Trận Nhầm Lẫn Chuẩn Hóa 17 Lớp (Test Acc: {test_acc*100:.2f}%)', fontsize=14, fontweight='bold', pad=15)
plt.xlabel('Nhãn Dự Đoán (Predicted)', fontsize=12)
plt.ylabel('Nhãn Thực Tế (True)', fontsize=12)
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig("confusion_matrix_efficientnetb0.png", dpi=150)
plt.show()

In [ ]:
# CELL 10: Test Set Inference Demonstration with Latency & Grad-CAM Heatmap
def make_gradcam_heatmap(img_array, model, last_conv_layer_name="top_conv"):
    try:
        base = model.get_layer("efficientnetb0")
        grad_model = tf.keras.models.Model(
            inputs=[base.input],
            outputs=[base.get_layer(last_conv_layer_name).output, model(base.input)]
        )
    except Exception:
        grad_model = tf.keras.models.Model(
            inputs=[model.inputs],
            outputs=[model.get_layer("top_conv").output, model.output]
        )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        top_class = tf.argmax(predictions[0])
        score = predictions[:, top_class]

    grads = tape.gradient(score, conv_outputs)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_outputs = conv_outputs[0]
    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.math.reduce_max(heatmap) + 1e-10)
    return heatmap.numpy()

def overlay_heatmap(orig_img, heatmap, alpha=0.45):
    h, w = orig_img.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h))
    hm_uint8 = np.uint8(255 * hm_resized)
    hm_colored = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_rgb = cv2.cvtColor(hm_colored, cv2.COLOR_BGR2RGB)
    blended = cv2.addWeighted(orig_img, 1.0 - alpha, hm_rgb, alpha, 0)
    return blended

# Bốc ngẫu nhiên 12 ảnh từ tập Test để chạy suy luận
test_files_flat = test_generator.filepaths
sample_indices = np.random.choice(len(test_files_flat), size=12, replace=False)

plt.figure(figsize=(18, 14))
latencies = []

for idx_plot, s_idx in enumerate(sample_indices):
    img_path = test_files_flat[s_idx]
    true_label = all_classes[test_generator.classes[s_idx]]
    
    raw_img = cv2.imread(img_path)
    raw_rgb = cv2.cvtColor(raw_img, cv2.COLOR_BGR2RGB)
    resized_rgb = cv2.resize(raw_rgb, (IMG_WIDTH, IMG_HEIGHT))
    
    input_tensor = efficientnet_preprocess_input(resized_rgb.copy().astype(np.float32))
    input_tensor = np.expand_dims(input_tensor, axis=0)
    
    # Đo thời gian suy luận
    t0 = time.perf_counter()
    preds = model.predict(input_tensor, verbose=0)[0]
    t1 = time.perf_counter()
    latencies.append((t1 - t0) * 1000)
    
    pred_idx = np.argmax(preds)
    pred_label = all_classes[pred_idx]
    confidence = preds[pred_idx] * 100
    
    # Tính Grad-CAM
    try:
        heatmap = make_gradcam_heatmap(input_tensor, model)
        vis_img = overlay_heatmap(resized_rgb, heatmap, alpha=0.45)
    except Exception:
        vis_img = resized_rgb
        
    ax = plt.subplot(3, 4, idx_plot + 1)
    ax.imshow(vis_img)
    is_correct = (true_label == pred_label)
    title_color = 'green' if is_correct else 'red'
    
    ax.set_title(
        f"Thực tế: {true_label}\nDự đoán: {pred_label}\nĐộ tin cậy: {confidence:.1f}%",
        fontsize=9, fontweight='bold', color=title_color
    )
    ax.axis('off')

avg_latency = np.mean(latencies)
fps = 1000.0 / avg_latency if avg_latency > 0 else 0
print(f"\n=================================================================")
print(f"   HIỆU NĂNG SUY LUẬN THỰC TẾ TRÊN TẬP TEST:")
print(f"   - Độ trễ trung bình : {avg_latency:.2f} ms / ảnh")
print(f"   - Tốc độ xử lý       : {fps:.1f} FPS")
print(f"=================================================================\n")

plt.suptitle(f"Trực Quan Hóa Suy Luận 12 Ảnh Test (Xanh: Đúng | Đỏ: Sai) - Latency: {avg_latency:.1f}ms", fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig("test_inferences_gradcam.png", dpi=150)
plt.show()

In [ ]:
# CELL 11: Export Model to Lightweight TFLite for Deployment
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

tflite_model = converter.convert()

tflite_path = "models/EfficientNetB0_model.tflite"
with open(tflite_path, "wb") as f:
    f.write(tflite_model)

labels_path = "models/EfficientNetB0_model_labels.txt"
with open(labels_path, "w", encoding="utf-8") as f:
    for cls in all_classes:
        f.write(f"{cls}\n")

file_size_mb = os.path.getsize(tflite_path) / (1024 * 1024)
print("=================================================================")
print("   XUẤT MÔ HÌNH TFLITE THÀNH CÔNG CHO TRIỂN KHAI THỰC TẾ")
print(f"   - File mô hình TFLite : {tflite_path} ({file_size_mb:.2f} MB)")
print(f"   - File nhãn 17 lớp     : {labels_path}")
print("=================================================================")